# MergeDNA Implementation
This notebook demonstrates an implementation of the [MergeDNA paper](https://arxiv.org/pdf/2511.14806).

This implementation was written completely in pytorch and includes:
- The MergeDNA architecture:
    - Local Encoder with windowed attention and windowed DTEM.
    - Latent Encoder with global attention and BSM for the latent reconstruction task.
    - Latent Decoder with global attention.
    - Local Decoder with windowed attention.
- Multi-objective pretraining on the NT Genome Multi-Species dataset:
    - Full sequence reconstruction from local encoder compressed embeddings.
    - Full sequence reconstruction from latent encoder compressed embeddings (frozen local encoder).
    - Adaptive masked token modelling derived from latent encoder source matrix.

### Limitations
- The full pre-training could not be completed due to computational limitations. Instead pretraining is done with 10,000 training samples on a model size of 14mil parameters, and validated on a small 1,000 sample dataset.
- Some of the pre-training implementation has been excluded to keep things simple. This is things like using an LR scheduler or varying the local encoder token reduction during training.
- With more time it would have been nice to demonstrate some fine-tuning and evaluation on downstream tasks.
- The readability of this code was prioritised over optimisation. Places that could be optimised have been marked
with a `# TODO[OptCand]` comment.
- This is not meant to be production-grade code, better documentation would be needed.
- It would have been preferable to use FlexAttention from pytorch but it was not compatible with a MacBook.
- `torch.compile` broke when BSM was added, it would be good to figure out why.
- More time could be spent validating the windowed BSM is grouping tokens locally.
- Figure out how the window mask should be updated based on source matrix, as order changes as merging occurs.

### Dependencies
This notebook was written with `Python 3.13`, using the libraries:
- `torch==2.10.0`
- `datasets==3.6.0`
- `bio==1.8.1`

In [1]:
# Import the following libraries
import math
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from typing import Optional

### Masked Token Modelling Tokenizer

In [2]:
# Tokenizer
class MtmDnaTokenizer:
    """A Masked Token Modelling DNA Tokenizer"""

    PAD: str = "[PAD]"
    UNK: str = "[UNK]"
    CLS: str = "[CLS]"
    SEP: str = "[SEP]"
    MASK: str = "[MASK]"
    ALL_TOKENS: list[str] = [PAD, UNK, CLS, SEP, MASK, "A", "C", "G", "T"]
    char_to_id: dict[str, int]
    id_to_char: dict[int, str]

    def __init__(self) -> None:
        self.char_to_id = dict(zip(self.ALL_TOKENS, range(len(self.ALL_TOKENS))))
        self.id_to_char = {v: k for k, v in self.char_to_id.items()}

    @property
    def vocab_size(self) -> int:
        return len(self.ALL_TOKENS)

    def encode(self, sequences: list[str], max_seq_len: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Encodes a batch of sequences in to token ids.
        The output sequence length with be +2 larger than the max_seq_len due to [CLS] and [SEP] tokens.

        Args:
            sequences: list[str], the batch raw sequence strings to be converted into token ids
            max_seq_len: int, the maximum length of the sequences.

        Returns:
            token_ids: Tensor[batch_size, sequence_len], The tokenized sequence batch.
        """
        PAD_ID = self.char_to_id[self.PAD]
        UNK_ID = self.char_to_id[self.UNK]
        CLS_ID = self.char_to_id[self.CLS]
        SEP_ID = self.char_to_id[self.SEP]
        token_ids = torch.full(size=(len(sequences), max_seq_len + 2), fill_value=PAD_ID, dtype=torch.long)
        pad_masks = torch.ones(size=(len(sequences), max_seq_len + 2), dtype=torch.bool)

        for i, seq in enumerate(sequences):
            id_gen = (self.char_to_id.get(char, UNK_ID) for _, char in zip(range(max_seq_len), seq, strict=False))
            token_ids[i, : len(seq) + 2] = torch.tensor([CLS_ID, *id_gen, SEP_ID])
            pad_masks[i, : len(seq) + 2] = False

        return token_ids, pad_masks

    def decode(self, token_ids: torch.Tensor, seq_lengths: torch.Tensor) -> list[str]:
        """Decodes a batch of predicted token ids into raw strings.

        Args:
            token_ids: Tensor[batch_size, sequence_len], The batch of predicted token ids.
            seq_lengths: Tensor[batch_size], The lengths of each sequence in the batch.

        Returns:
            token_ids: Tensor[batch_size, sequence_len]
        """
        sequences = []
        for seq_ids, seq_len in zip(token_ids, seq_lengths):
            sequences.append("".join((self.id_to_char[int(token_id)] for token_id in seq_ids[1 : seq_len + 1])))
        return sequences

### Dataset

In [3]:
# Dataset class
class NtMsGenomeDataset(Dataset):
    """The Nucleotide-Transformer Multi-Species Genome Dataset"""

    token_ids: torch.Tensor
    pad_masks: torch.Tensor
    seq_lengths: torch.Tensor
    tokenizer: MtmDnaTokenizer

    def __init__(self, samples: list[str], max_seq_len: int, tokenizer: MtmDnaTokenizer) -> None:
        """
        Args:
            samples: list[str], the raw sequences for the dataset
            tokenizer: MtmDnaTokenizer, the tokenizer to encode the sequences into token ids.
        """
        super().__init__()
        self.token_ids, self.pad_masks = tokenizer.encode(samples, max_seq_len)
        self.seq_lengths = torch.tensor(list(map(lambda x: min(len(x), max_seq_len), samples)))
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        return len(self.token_ids)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.token_ids[idx], self.pad_masks[idx], self.seq_lengths[idx]

### Masking

In [4]:
# Masking
def apply_masking(
    token_ids: torch.Tensor, seq_lengths: torch.Tensor, latent_s: torch.Tensor, mask_id: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Uses Masked Token Modelling (MTM) to randomly mask the sequence based on the latent source tokens.
    The latent source tokens are used to derive the probabilities of sampling a token for masking without replacement.
    Of the token ids, 15% will be selected for maskign, then 80% of those are masked, 10% are swapped, 10% kept.
    
    Args:
        token_ids: Tensor[batch_size, seq_len], The batch of sequence tokens to mask.
        seq_lengths: Tensor[batch_size], The lengths of each sequence in the batch.

    Returns:
        masked_ids: Tensor[batch_size, seq_len], The masked token ids.
        mask_matrix: Tensor[batch_size, seq_len], The binary matrix indicating which tokens have been masked.
    """
    token_probs = (latent_s.mT @ latent_s.sum(dim=-1, keepdim=True))[..., 0]
    token_probs = 1 / token_probs**2
    token_probs = token_probs / token_probs.norm(dim=-1, keepdim=True)
    masked_ids = token_ids.clone()
    mask_matrix = torch.zeros_like(masked_ids, dtype=torch.bool)
    for i in range(len(masked_ids)):
        seq_len = int(seq_lengths[i])
        select_inds = torch.multinomial(token_probs[i][1 : seq_len + 1], int(seq_len * 0.15), replacement=False)
        mask_matrix[i, select_inds + 1] = True
        mask_inds = select_inds[: int(len(select_inds) * 0.8)]  # First 80% of selected tokens get masked
        swap_inds = select_inds[int(len(select_inds) * 0.9) :]  # Last 10% of selected tokens get swapped
        masked_ids[i, mask_inds] = mask_id
        masked_ids[i, swap_inds] = token_ids[i, torch.roll(swap_inds, shifts=[1], dims=[0])]

    return masked_ids, mask_matrix

### Rotory Embeddings

In [5]:
# Rotory Embeddings
def rotate_embeddings(x: torch.Tensor) -> torch.Tensor:
    """Uses the RoPE mechanism to apply a pairwise rotation based on sequence positions.

    Args:
        x: Tensor[batch_size, num_heads, sequence_len, embedding_dims], the sequence embeddings.

    Returns:
        x: Shape[batch_size, num_heads, sequence_len, embedding_dims], the rotated sequence embeddings.
    """
    _, _, S, D = x.shape
    i = x.new_tensor(range(D // 2)).repeat_interleave(2)
    theta = torch.pow(10_000, -2 * i / D)
    positions = x.new_tensor(range(S))
    outer = torch.outer(positions, theta)
    cos = torch.cos(outer)
    sin = torch.sin(outer)
    xflip = torch.stack([-x[..., 1::2], x[..., ::2]], dim=-1).flatten(-2, -1)
    return x * cos + xflip * sin

### Token Merging (ToMe)

In [6]:
# Vanilla Token Merging (ToMe)
@torch.no_grad
def bipartite_soft_matching(
    x: torch.Tensor,
    m: torch.Tensor,
    p: torch.Tensor,
    s: torch.Tensor,
    reduction: int,
    window_size: Optional[int] = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Applied BSM to the input sequence, merging sequence embeddings based on their similarity embeddings.
    The number of tokens in the output sequence will be reduced by the number defined by `reduction`.

    Args:
        x: Tensor[batch_size, reduced_len, embedding_dims], The sequence embeddings.
        m: Tensor[batch_size, reduced_len, embedding_dims], The matching similarity embeddings.
        p: Tensor[batch_size, reduced_len], The binary padding mask for the sequence.
        s: Tensor[batch_size, reduced_len, sequence_len]: The binary source matrix to track the token merging from its original sequence.
        reduction: int, the amount of tokens to merge into others.
        window_size: optional[int], if enabled, ensures matching is only done within the token window size.

    Returns:
        x: Tensor[batch_size, reduced_len, embedding_dims], The merged sequence embeddings.
        p: Tensor[batch_size, reduced_len], The updated binary padding mask for the sequence.
        s: Tensor[batch_size, reduced_len, sequence_len], The updated binary source matrix.
    """
    # Split similarity embeddings and feature embeddings
    ma, mb = m[:, ::2, :], m[:, 1::2, :]
    xa, xb = x[:, ::2, :], x[:, 1::2, :]
    pa, pb = p[:, ::2], p[:, 1::2]
    sa, sb = s[:, ::2, :], s[:, 1::2, :]

    # Calculate cosine similarity between sets A and B
    ma = ma / ma.norm(dim=-1, keepdim=True)
    mb = mb / mb.norm(dim=-1, keepdim=True)
    sim_mat = ma @ mb.mT
    block_mask = pa[0].bool().unsqueeze(1) | pb[0].bool()

    # Create window mask
    if window_size is not None:
        indices = x.new_tensor(range(x.size(1)))
        window_mask = (indices.view(-1, 1) - indices).abs() > window_size
        # TODO: Figure out how to update the window mask based on the adjacency matrix
        block_mask = block_mask | window_mask[::2, 1::2]

    # TODO[OptCand]: This is a very sparse matrix, avoid computation on all the zeros
    sim_mat = sim_mat.masked_fill(block_mask, float("-inf"))  # Ensure padding doesn't merge

    # Find closest B tokens for each A token, then keep top k
    B, S, E = xa.shape
    b_max_sims, b_max_inds = torch.max(sim_mat, dim=-1)
    _, topk_a_inds = torch.topk(b_max_sims, reduction)
    topk_b_inds = torch.gather(b_max_inds, dim=-1, index=topk_a_inds)
    adj = torch.zeros_like(sim_mat, dtype=torch.int)  # Defines the adjacency matrix
    adj[torch.arange(B)[:, None], topk_a_inds, topk_b_inds] = True

    # Merge top k A tokens in B, weighted by number of tokens A represents
    xa_norm = sa.sum(dim=-1, keepdim=True)
    b_norm = adj.sum(dim=-2).clamp(min=1)[..., None]
    xb = xb + (adj.float().mT @ (xa / xa_norm)) / b_norm
    sb = sb + (adj.float().mT @ sa)

    # Removed merged A tokens and reconstruct sequence
    ua_mask = adj.sum(dim=-1) == 0  # Unassigned tokens of A set
    xa = xa[ua_mask].view(B, S - reduction, E)
    pa = pa[ua_mask].view(B, S - reduction)
    sa = sa[ua_mask].view(B, S - reduction, s.size(-1))
    x = torch.cat([xb, xa], dim=1)
    p = torch.cat([pb, pa], dim=1)
    s = torch.cat([sb, sa], dim=1)

    return x, p, s


@torch.no_grad
def unmerge(x: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
    """Unmerges BSM merged tokens back to their original sequence.

    Args:
        x: Tensor[batch_size, reduced_len, embedding_dims], The merged sequence embeddings.
        s: Tensor[batch_size, reduced_len, sequence_len]: The binary source matrix to track the token merging from its original sequence .

    Returns:
        x: Tensor[batch_size, sequence_len, embedding_dims], The unmerged sequence embeddings.
    """
    return s.mT @ x

### Windowed DTEM

In [7]:
# Windowed Differentiable Token Merging (DTEM)
class WindowedDtemLayer(torch.nn.Module):
    """The Windowed Differentiable Token Merging layer"""

    xproj: torch.nn.Linear

    def __init__(self, embedding_dims: int, window_size: int, top_k: int, temperature: float) -> None:
        """
        Args:
            embedding_dims: int, The size of the embedding dimension of the input sequences.
            window_size: int, The amount of tokens to consider during merging around each token.
            top_k: int, The number of tokens from set B to be considered during soft grouping.
            temperature: float, Tuning parameter for regulating relaxation.
        """
        super().__init__()
        self.window_size = window_size
        self.top_k = top_k
        self.temperature = temperature
        self.xproj = torch.nn.Linear(embedding_dims, embedding_dims, bias=False)

    def forward(
        self,
        x: torch.Tensor,
        x_att: torch.Tensor,
        p: torch.Tensor,
        s: torch.Tensor,
        reduction: int,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Applies windowed DTEM to the attention scores based on decoupled similarity scores.
        This uses BSM when doing evaluation, which will cause the sequence lengths to reduce.
        When training this applies soft merging so the sequence lengths never actually change, the embeddings are just redistributed.

        Args:
            x: Tensor[batch_size, reduced_len, embedding_dims], the pre-attention token embeddings.
            x_att: Tensor[batch_size, reduced_len, embedding_dims], the post-attention token embeddings.
            p: Tensor[batch_size, reduced_len], The binary padding mask for the sequence.
            s: Tensor[batch_size, reduced_len, sequence_len], The binary source matrix.
            reduction: The number of times to apply relaxation during soft grouping.

        Returns:
            x: Tensor[batch_size, reduced_len, embedding_dims], the merged token embeddings.
            p: Tensor[batch_size, reduced_len], The updated binary padding mask for the sequence.
            s: Tensor[batch_size, reduced_len, sequence_len], The updated binary source matrix.
        """
        m = self.xproj(x)
        if not self.training:
            x, p, s = bipartite_soft_matching(x_att, m, p, s, reduction, self.window_size)
            return x, p, s

        # Split similarity embeddings and feature embeddings
        ma, mb = m[:, ::2, :], m[:, 1::2, :]
        xa, xb = x[:, ::2, :], x[:, 1::2, :]
        pa, pb = p[:, ::2], p[:, 1::2]
        sa, sb = s[:, ::2, :], s[:, 1::2, :]

        # Create window mask
        indices = x.new_tensor(range(x.size(1)))
        window_mask = (indices.view(-1, 1) - indices).abs() > self.window_size

        # Calculate cosine similarity between sets A and B
        ma = ma / ma.norm(dim=-1, keepdim=True)
        mb = mb / mb.norm(dim=-1, keepdim=True)
        sim_mat = ma @ mb.mT
        # TODO[OptCand]: This is a very sparse matrix, avoid computation on all the zeros
        block_mask = (pa[0].bool().unsqueeze(1) | pb[0].bool()) | window_mask[::2, 1::2]
        sim_mat = sim_mat.masked_fill(block_mask, float("-inf"))  # Ensure padding doesn't merge

        # Find closest B tokens for each A token, then keep top k
        topk_idx = sim_mat.argsort(dim=-1, descending=True)[..., : self.top_k]
        topk_sim = sim_mat.gather(dim=-1, index=topk_idx)
        B, S, K = topk_sim.shape

        # Use soft merging to derive approximate adjacency matrix
        adj = torch.zeros_like(topk_sim)
        for _ in range(reduction):
            soft = F.softmax((topk_sim / self.temperature).view(B, -1), dim=-1).view(B, S, K)
            topk_sim = topk_sim + torch.log(1 - soft.sum(dim=-1, keepdim=True))
            adj = adj + soft

        norm = adj.sum(dim=-1, keepdim=True)
        adj = adj / norm.detach().clamp(min=1)
        adj = torch.zeros_like(sim_mat).scatter_reduce(-1, topk_idx, adj, reduce="sum")

        # Soft merge top k A tokens in B, weighted by number of tokens A represents
        esa = sa.sum(dim=-1, keepdim=True)
        esb = sb.sum(dim=-1, keepdim=True)
        next_sb = sb + adj.mT @ sa
        xb = (esb * xb + adj.mT @ (esa * xa)) / next_sb.sum(axis=-1, keepdim=True)
        sb = next_sb
        sa = sa * (1 - adj.sum(axis=-1, keepdim=True))

        # Removed merged A tokens and reconstruct sequence
        x = torch.cat([xb, xa], dim=1)
        s = torch.cat([sb, sa], dim=1)

        return x, p, s

### Windowed Attention

In [8]:
# Windowed Attention
class WindowedAttentionLayer(torch.nn.Module):
    """The windowed attention layer."""

    num_heads: int
    heads_dim: int
    window_size: int
    qkv: torch.nn.Linear
    out_proj: torch.nn.Linear

    def __init__(self, embedding_dims: int, num_heads: int, window_size: int) -> None:
        """
        Args:
            embedding_dims: int, The size of the embedding dimension of the input sequences.
            num_heads: int, The number of attention heads to use during attention
            window_size: int, Ensures attention is only done within the token window size.
        """
        super().__init__()
        self.num_heads = num_heads
        self.heads_dim = embedding_dims // num_heads
        self.window_size = window_size
        self.qkv = torch.nn.Linear(embedding_dims, embedding_dims * 3, bias=False)
        self.out_proj = torch.nn.Linear(embedding_dims, embedding_dims, bias=False)

    def forward(self, x: torch.Tensor, p: torch.Tensor, s: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Applies global attention weighted by the binary source matrix.

        Args:
            x: Tensor[batch_size, reduced_len, embedding_dims], sequence embeddings.
            p: Tensor[batch_size, reduced_len], The binary padding mask for the sequence.
            s: Tensor[batch_size, reduced_len, sequence_len], The binary source matrix.

        Returns:
            x: Tensor[batch_size, reduced_len, embedding_dims], the updated sequence embeddings.
        """
        B, S, E = x.size()
        # q, k, v shapes [batch_size, num_heads, seq_len, embed_depth]
        q, k, v = self.qkv(x).view(B, S, 3, self.num_heads, self.heads_dim).permute(2, 0, 3, 1, 4).unbind(0)
        q, k = rotate_embeddings(q), rotate_embeddings(k)
        att = (q @ k.mT) / math.sqrt(E)
        if s is not None:
            att = att + torch.log(s.sum(dim=-1)).view(B, 1, 1, S)

        # Create window mask
        indices = x.new_tensor(range(S))
        window_mask = (indices.view(-1, 1) - indices).abs() > self.window_size

        # TODO[OptCand]: This is a very sparse matrix, avoid computation on all the zeros
        att = att.masked_fill(window_mask | p.unsqueeze(1).unsqueeze(2), float("-inf"))
        att = F.softmax(att, dim=-1) @ v
        out = att.transpose(1, 2).contiguous().view(B, S, E)
        return self.out_proj(out)

### Global Attention

In [9]:
# Global Attention
class GlobalAttentionLayer(torch.nn.Module):
    """The global attention layer."""

    num_heads: int
    heads_dim: int
    qkv: torch.nn.Linear
    out_proj: torch.nn.Linear

    def __init__(self, embedding_dims: int, num_heads: int) -> None:
        """
        Args:
            embedding_dims: int, The size of the embedding dimension of the input sequences.
            num_heads: int, The number of attention heads to use during attention
        """
        super().__init__()
        self.num_heads = num_heads
        self.heads_dim = embedding_dims // num_heads
        self.qkv = torch.nn.Linear(embedding_dims, embedding_dims * 3, bias=False)
        self.out_proj = torch.nn.Linear(embedding_dims, embedding_dims, bias=False)

    def forward(self, x: torch.Tensor, p: torch.Tensor, s: torch.Tensor) -> torch.Tensor:
        """Applies global attention weighted by the binary source matrix.

        Args:
            x: Tensor[batch_size, reduced_len, embedding_dims], sequence embeddings.
            p: Tensor[batch_size, reduced_len], The binary padding mask for the sequence.
            s: Tensor[batch_size, reduced_len, sequence_len], The binary source matrix.

        Returns:
            x: Tensor[batch_size, reduced_len, embedding_dims], the updated sequence embeddings.
        """
        B, S, E = x.size()
        # q, k, v shapes [batch_size, num_heads, seq_len, embed_depth]
        q, k, v = self.qkv(x).view(B, S, 3, self.num_heads, self.heads_dim).permute(2, 0, 3, 1, 4).unbind(0)
        q, k = rotate_embeddings(q), rotate_embeddings(k)
        att = ((q @ k.mT) / math.sqrt(E)) + torch.log(s.sum(dim=-1)).view(B, 1, 1, S)
        att = att.masked_fill(p.unsqueeze(1).unsqueeze(2), float("-inf")) # TODO[OptCand]: FlashAttention
        att = F.softmax(att, dim=-1) @ v
        out = att.transpose(1, 2).contiguous().view(B, S, E)
        return self.out_proj(out)

### FFN + SwiGLU

In [10]:
# FFN + SwiGLU
class FfnSwiGluLayer(torch.nn.Module):
    """A Feed-Forward Layer using the SwiGLU activation function."""

    up_proj_gated: torch.nn.Linear
    down_proj: torch.nn.Linear

    def __init__(self, embedding_dims: int) -> None:
        """
        Args:
            embedding_dims: int, The size of the embedding dimension of the input sequences.
        """
        super().__init__()
        hidden_depth = int(embedding_dims * 8 / 3)
        self.up_proj_gated = torch.nn.Linear(embedding_dims, hidden_depth * 2, bias=False)
        self.down_proj = torch.nn.Linear(hidden_depth, embedding_dims, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Applies fanned linear projection with a SwiGLU activation between projections.
        Args:
            x: Tensor[batch_size, reduced_len, embedding_dims], sequence embeddings.

        Returns:
            x: Tensor[batch_size, reduced_len, embedding_dims], the updated sequence embeddings.
        """
        out, g = self.up_proj_gated(x).chunk(2, dim=-1)
        out = F.silu(out) * g
        return self.down_proj(out)

### Local Encoder

In [11]:
# Local Encoder
class LocalEncoderBlock(torch.nn.Module):
    """A local encoder block."""

    norm: torch.nn.RMSNorm
    att: WindowedAttentionLayer
    dtem: WindowedDtemLayer
    ffn: FfnSwiGluLayer

    def __init__(self, embedding_dims: int, num_heads: int, window_size: int, top_k: int, temperature: float) -> None:
        """
        Args:
            embedding_dims: int, The size of the embedding dimension of the input sequences.
            num_heads: int, The number of heads to use for windowed attention
            window_size: int, The amount of tokens around each token to consider during attention and merging.
            top_k: int, The number of tokens from set B to be considered during soft grouping.
            temperature: float, Tuning parameter for regulating DTEM relaxation.
        """
        super().__init__()
        self.norm = torch.nn.RMSNorm(embedding_dims)
        self.att = WindowedAttentionLayer(embedding_dims, num_heads, window_size)
        self.dtem = WindowedDtemLayer(embedding_dims, window_size, top_k, temperature)
        self.ffn = FfnSwiGluLayer(embedding_dims)

    def forward(
        self, x: torch.Tensor, p: torch.Tensor, s: torch.Tensor, reduction: int
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Applies the mergeDNA local encoder transformation to the sequence embeddings.

        Args:
            x: Tensor[batch_size, reduced_len, embedding_dims], sequence embeddings.
            p: Tensor[batch_size, reduced_len], The binary padding mask for the sequence.
            s: Tensor[batch_size, reduced_len, sequence_len], The binary source matrix.
            reduction: then number of tokens being merged in this block.

        Returns:
            x: Tensor[batch_size, reduced_len, embedding_dims], the updated sequence embeddings.
            p: Tensor[batch_size, reduced_len], The updated binary padding mask for the sequence.
            s: Tensor[batch_size, reduced_len, sequence_len], The updated binary source matrix.
        """
        x_att = x + self.att(self.norm(x), p, s)
        x_att, p, s = self.dtem(x, x_att, p, s, reduction)
        return x_att + self.ffn(self.norm(x_att)), p, s

### Latent Encoder/Decoder

In [12]:
# Latent Encoder/Decoder
class LatentBlock(torch.nn.Module):
    """A latent encoder/decoder block."""

    norm: torch.nn.RMSNorm
    att: GlobalAttentionLayer
    ffn: FfnSwiGluLayer

    def __init__(self, embedding_dims: int, num_heads: int) -> None:
        """
        Args:
            embedding_dims: int, The size of the embedding dimension of the input sequences.
            num_heads: int, The number of heads to use for windowed attention
        """
        super().__init__()
        self.num_heads = num_heads
        self.norm = torch.nn.RMSNorm(embedding_dims)
        self.att = GlobalAttentionLayer(embedding_dims, num_heads)
        self.ffn = FfnSwiGluLayer(embedding_dims)

    def forward(
        self,
        x: torch.Tensor,
        p: torch.Tensor,
        s: Optional[torch.Tensor] = None,
        reduction: Optional[int] = None,
    ) -> tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor]]:
        """Applies the mergeDNA latent block transformation to the sequence embeddings.

        Args:
            x: Tensor[batch_size, reduced_len, embedding_dims], sequence embeddings.
            p: Tensor[batch_size, reduced_len], The binary padding mask for the sequence.
            s: Tensor[batch_size, reduced_len, sequence_len], The binary source matrix. Not included if doing decoding.
            reduction: the number of tokens to merge.

        Returns:
            x: Tensor[batch_size, reduced_len, embedding_dims], the updated sequence embeddings.
            p: Tensor[batch_size, reduced_len], The updated binary padding mask for the sequence.
            s: Tensor[batch_size, reduced_len, sequence_len], The updated binary source matrix.
        """
        x = x + self.att(self.norm(x), p, s)
        if s is not None and reduction is not None:
            x, p, s = bipartite_soft_matching(x, x, p, s, reduction)
        return x + self.ffn(self.norm(x)), p, s

### Local Decoder

In [13]:
# Local Decoder
class LocalDecoderBlock(torch.nn.Module):
    """A local encoder block."""

    norm: torch.nn.RMSNorm
    att: WindowedAttentionLayer
    ffn: FfnSwiGluLayer

    def __init__(self, embedding_dims: int, num_heads: int, window_size: int) -> None:
        """
        Args:
            embedding_dims: int, The size of the embedding dimension of the input sequences.
            num_heads: int, The number of heads to use for windowed attention
            window_size: int, The amount of tokens around each token to consider during attention and merging.
        """
        super().__init__()
        self.norm = torch.nn.RMSNorm(embedding_dims)
        self.att = WindowedAttentionLayer(embedding_dims, num_heads, window_size)
        self.ffn = FfnSwiGluLayer(embedding_dims)

    def forward(self, x: torch.Tensor, p: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Applies the mergeDNA local decoder transformation to the sequence embeddings.

        Args:
            x: Tensor[batch_size, sequence_len, embedding_dims], sequence embeddings.
            p: Tensor[batch_size, sequence_len], The binary padding mask for the sequence.

        Returns:
            x: Tensor[batch_size, sequence_len, embedding_dims], the updated sequence embeddings.
        """
        x = x + self.att(self.norm(x), p)
        return x + self.ffn(self.norm(x))

### MergeDNA Model

In [14]:
# MergeDNA model
class MergeDNAModel(torch.nn.Module):
    embedder: torch.nn.Embedding
    local_encoder: torch.nn.ModuleList
    latent_encoder: torch.nn.ModuleList
    latent_decoder: torch.nn.ModuleList
    local_decoder: torch.nn.ModuleList
    head: torch.nn.Linear

    def __init__(
        self,
        local_encoder_blocks: int,
        latent_encoder_blocks: int,
        latent_decoder_blocks: int,
        local_decoder_blocks: int,
        vocab_size: int,
        embedding_dims: int,
        num_heads: int,
        window_size: int,
        top_k: int,
        temperature: float,
    ) -> None:
        """
        Args:
            local_encoder_blocks: int, The number of local encoder blocks to use in the model.
            latent_encoder_blocks: int, The number of latent encoder blocks to use in the model.
            latent_decoder_blocks: int, The number of latent decoder blocks to use in the model.
            local_decoder_blocks: int, The number of local decoder blocks to use in the model.
            vocab_size: The number of tokens in the model vocabulary.
            embedding_dims: int, The size of the embedding dimension of the input sequences.
            num_heads: int, The number of heads to use for windowed attention
            window_size: int, The amount of tokens around each token to consider during attention and merging.
            top_k: int, The number of tokens from set B to be considered during soft grouping.
            temperature: float, Tuning parameter for regulating DTEM relaxation.
        """
        super().__init__()
        self.embedder = torch.nn.Embedding(vocab_size, embedding_dims)
        self.local_encoder = torch.nn.ModuleList(
            [
                LocalEncoderBlock(embedding_dims, num_heads, window_size, top_k, temperature)
                for _ in range(local_encoder_blocks)
            ]
        )
        self.latent_encoder = torch.nn.ModuleList(
            [LatentBlock(embedding_dims, num_heads) for _ in range(latent_encoder_blocks)]
        )
        self.latent_decoder = torch.nn.ModuleList(
            [LatentBlock(embedding_dims, num_heads) for _ in range(latent_decoder_blocks)]
        )
        self.local_decoder = torch.nn.ModuleList(
            [LocalDecoderBlock(embedding_dims, num_heads, window_size) for _ in range(local_decoder_blocks)]
        )
        self.head = torch.nn.Linear(embedding_dims, vocab_size, bias=False)

    def forward(
        self,
        token_ids: torch.Tensor,
        p: torch.Tensor,
        local_reduction: int,
        latent_reduction: Optional[int] = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Applies the forward pass of the mergeDNA model.

        Args:
            token_ids: Tensor[batch_size, sequence_len], sequence token ids.
            p: Tensor[batch_size, sequence_len], The binary padding mask for the sequence.
            local_reduction: int, The number of local tokens to merge per local encoder block.
            latent_reduction: Optional[int], if enabled, the number of latent tokens to merge per latent encoder block.

        Returns:
            x: Tensor[batch_size, sequence_len, embedding_dims], the updated sequence embeddings.
            latent_s: Tensor[batch_size, sequence_len, sequence_len], the binary source matrix from the latent encoder.
        """
        B, S = token_ids.size()
        s = p.new_zeros(B, S, S, dtype=torch.float)  # Tensor[batch_size, sequence_len, sequence_len]
        s.diagonal(dim1=-2, dim2=-1).fill_(1.0)

        # Local Encoder Forward
        with torch.set_grad_enabled(latent_reduction is None):
            x = self.embedder(token_ids)
            local_p, local_s = p, s
            for block in self.local_encoder:
                x, local_p, local_s = block(x, local_p, local_s, local_reduction)

        # Latent Encoder Forward
        latent_p, latent_s = local_p, local_s
        for block in self.latent_encoder:
            x, latent_p, latent_s = block(x, latent_p, latent_s, latent_reduction)

        if latent_reduction is not None:
            x = unmerge(x, latent_s)

        # Latent Decoder Forward
        for block in self.latent_decoder:
            x, _, _ = block(x, local_p, local_s)

        x = unmerge(x, local_s)

        # # Local Decoder Forward
        for block in self.local_decoder:
            x = block(x, p)

        return self.head(x), latent_s


### Dataset Loading

In [15]:
# Load datasets
hf_train = load_dataset("InstaDeepAI/multi_species_genomes", trust_remote_code=True, split="train", streaming=True)
hf_val = load_dataset("InstaDeepAI/multi_species_genomes", trust_remote_code=True, split="validation", streaming=True)
train_samples = list(map(lambda x: x["sequence"], hf_train.take(10_000)))
val_samples = list(map(lambda x: x["sequence"], hf_train.take(1_000)))
print(f"Downloaded {len(train_samples)} training samples, and {len(val_samples)} validation samples.")

Downloaded 10000 training samples, and 1000 validation samples.


### Training and Validation

In [17]:
# Training and validation
seed = 123
epochs = 2
batches_per_epoch = 10
epochs_per_val = 1
train_batch_size = 64
val_batch_size = 256
sequence_len = 256
local_reduction = 64
latent_reduction = 16
lr = 1e-4
lerl_weight = 0.25
accelerator = "mps"  # cpu, gpu, mps

torch.manual_seed(seed)

# Validated that when full model is created it has 380mil parameters
tokenizer = MtmDnaTokenizer()
model = MergeDNAModel(
    local_encoder_blocks=2,
    latent_encoder_blocks=10,
    latent_decoder_blocks=4,
    local_decoder_blocks=2,
    vocab_size=tokenizer.vocab_size,
    embedding_dims=256,
    num_heads=4,
    window_size=16,
    top_k=3,
    temperature=0.1,
)

# Total parameters (including frozen ones)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

# model = torch.compile(model) # TODO: work out why this stopped working
device = torch.device("cuda") if accelerator == "gpu" else torch.device(accelerator)
model = model.to(device)
optim = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=1e-8)

train_dataset = NtMsGenomeDataset(train_samples, sequence_len, tokenizer)
val_dataset = NtMsGenomeDataset(val_samples, sequence_len, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=val_batch_size, shuffle=False)


def run_training():
    for epoch in range(1, epochs + 1):
        # Run training
        model.train()

        epoch_metrics = []
        for batch_idx, (label_ids, padding_mask, seq_lengths) in enumerate(train_loader):
            if batch_idx >= batches_per_epoch:
                break
            label_ids, padding_mask, seq_lengths = (
                label_ids.to(device),
                padding_mask.to(device),
                seq_lengths.to(device),
            )

            # Full Reconstruction loss
            # TODO: sample local reduction from normal dist to ensure 40% - 60% local reduction.
            logits, _ = model(label_ids, padding_mask, local_reduction=local_reduction)
            loss1 = F.cross_entropy(logits.flatten(0, 1), label_ids.flatten(0, 1)) / seq_lengths.sum()
            full_acc = (logits.argmax(dim=-1) == label_ids).float().mean()

            # Latent Reconstruction loss
            logits, latent_s = model(
                label_ids, padding_mask, local_reduction=local_reduction, latent_reduction=latent_reduction
            )
            loss2 = F.cross_entropy(logits.flatten(0, 1), label_ids.flatten(0, 1)) / seq_lengths.sum()
            latent_acc = (logits.argmax(dim=-1) == label_ids).float().mean()

            # MTM loss
            token_ids, mask_matrix = apply_masking(
                label_ids, seq_lengths, latent_s, tokenizer.char_to_id[tokenizer.MASK]
            )
            logits, _ = model(token_ids, padding_mask, local_reduction=local_reduction)
            loss3 = F.cross_entropy(logits[mask_matrix], label_ids[mask_matrix]) / mask_matrix.sum()
            mask_acc = (logits.argmax(dim=-1)[mask_matrix] == label_ids[mask_matrix]).float().mean()
            loss = loss1 + lerl_weight * loss2 + loss3

            epoch_metrics.append(
                {
                    "loss1": loss1.item(),
                    "loss2": loss2.item(),
                    "loss3": loss3.item(),
                    "total_loss": loss.item(),
                    "full_acc": full_acc.item(),
                    "latent_acc": latent_acc.item(),
                    "mask_acc": mask_acc.item(),
                    "samples": len(label_ids),
                }
            )
            print("Step " + ", ".join(f"{k}: {v:.5f}" for k, v in epoch_metrics[-1].items()))

            optim.zero_grad()
            loss.backward()
            optim.step()

        epoch_loss = sum(item["total_loss"] for item in epoch_metrics)
        epoch_acc = sum(item["mask_acc"] * item["samples"] for item in epoch_metrics)
        epoch_samples = sum(item["samples"] for item in epoch_metrics)
        print(f"Epoch loss: {epoch_loss:.5f}, mask_acc: {epoch_acc / epoch_samples:.2%} ")

        # Check validation
        if epoch % epochs_per_val != 0:
            continue

        #  Run validation
        val_metrics = []
        model.eval()
        for label_ids, padding_mask, seq_lengths in val_loader:
            label_ids, padding_mask, seq_lengths = (
                label_ids.to(device),
                padding_mask.to(device),
                seq_lengths.to(device),
            )
            with torch.no_grad():
                logits, _ = model(label_ids, padding_mask, local_reduction=local_reduction)
            loss = F.cross_entropy(logits.flatten(0, 1), label_ids.flatten(0, 1)) / seq_lengths.sum()
            full_acc = (logits.argmax(dim=-1) == label_ids).float().mean()
            val_metrics.append(
                {
                    "loss": loss.item(),
                    "acc": full_acc.item(),
                    "samples": len(label_ids),
                }
            )

        val_loss = sum(item["loss"] for item in val_metrics)
        val_acc = sum(item["acc"] * item["samples"] for item in val_metrics)
        val_samples = sum(item["samples"] for item in val_metrics)
        print(f"Val loss: {val_loss:.5f}, full_acc: {val_acc / val_samples:.2%}")


run_training()

Total Parameters: 14,286,848
Step loss1: 0.00013, loss2: 0.00017, loss3: 0.00089, total_loss: 0.00106, full_acc: 0.40528, latent_acc: 0.24509, mask_acc: 0.31291, samples: 64.00000
Step loss1: 0.00012, loss2: 0.00017, loss3: 0.00082, total_loss: 0.00098, full_acc: 0.41933, latent_acc: 0.26133, mask_acc: 0.42064, samples: 64.00000
Step loss1: 0.00010, loss2: 0.00016, loss3: 0.00073, total_loss: 0.00088, full_acc: 0.51108, latent_acc: 0.25509, mask_acc: 0.52426, samples: 64.00000
Step loss1: 0.00009, loss2: 0.00016, loss3: 0.00066, total_loss: 0.00079, full_acc: 0.66485, latent_acc: 0.26351, mask_acc: 0.61102, samples: 64.00000
Step loss1: 0.00008, loss2: 0.00016, loss3: 0.00059, total_loss: 0.00071, full_acc: 0.71197, latent_acc: 0.25394, mask_acc: 0.65419, samples: 64.00000
Step loss1: 0.00007, loss2: 0.00015, loss3: 0.00052, total_loss: 0.00063, full_acc: 0.70064, latent_acc: 0.26647, mask_acc: 0.66488, samples: 64.00000
Step loss1: 0.00006, loss2: 0.00015, loss3: 0.00045, total_loss: 